# AI 가계부 배틀 — Colab 백업 노트북

오늘 아침 로컬 파이썬 환경이 말을 안 들었나요? 괜찮습니다. 이 노트북은 여러분
컴퓨터에 아무것도 설치하지 않고, 브라우저만으로 오늘 실습을 처음부터 끝까지
그대로 따라갈 수 있게 만든 비상용 백업입니다.

위에서부터 순서대로 셀을 하나씩 실행하세요 (셀을 클릭하고 `Shift+Enter`).
이 노트북을 여는 사람이 오늘 혼자만은 아닙니다 — 막히면 옆자리나 강사에게
편하게 물어보세요.

> ⚠️ **행사 전 확인 필수**: 아래 `REPO_URL`은 아직 만들어지지 않은 공개
> 저장소 주소의 placeholder입니다. 실제 저장소가 준비되면 이 값을 반드시
> 교체하세요.


## 0막 — 환경 준비

로컬에서 쓰던 `uv`나 `streamlit`은 여기서 필요 없습니다. Colab에는 파이썬이
이미 설치되어 있으니, 저장소를 내려받고 패키지 몇 개만 설치하면 바로 시작할
수 있습니다.


In [ ]:
import os
import sys

# ⚠️ 아직 만들어지지 않은 placeholder 주소입니다.
# 행사 전 실제 공개 저장소 주소로 반드시 교체하세요.
REPO_URL = "https://github.com/mindori/ai-budget-battle.git"

!git clone -q {REPO_URL} ai-budget-battle
%cd ai-budget-battle
sys.path.insert(0, os.getcwd())


In [ ]:
!pip install -q google-genai pydantic python-dotenv


API 키를 입력하세요. `getpass`를 쓰기 때문에 입력한 값이 화면에 그대로
보이지 않고, 이 노트북 파일에도 저장되지 않습니다. 키가 없다면
https://aistudio.google.com/apikey 에서 무료로 발급받을 수 있습니다
(신용카드 불필요).


In [ ]:
from getpass import getpass

api_key = getpass("Google API 키를 붙여넣고 Enter: ").strip()
if not api_key:
    raise ValueError("API 키가 비어 있습니다. 위 셀을 다시 실행해 키를 입력하세요.")
os.environ["GOOGLE_API_KEY"] = api_key


In [ ]:
def show_api_hint(error: Exception) -> None:
    """AI 호출이 실패했을 때 다음 행동을 안내한다."""
    print(
        "AI 호출에 실패했습니다.\n"
        "- 개인 핫스팟 등 다른 네트워크로 바꿔서 이 셀을 다시 실행해 보세요.\n"
        "- 위 API 키 입력 셀을 다시 실행해 키를 새로 넣어보세요.\n"
        "- 계속 실패하면 강사에게 도움을 요청하세요.\n"
        f"(원인: {error})"
    )


## 1막 — 본다: 영수증을 읽는다

영수증 사진 한 장을 Vision(멀티모달) LLM에 넣어 항목·금액·카테고리로
이루어진 구조화된 데이터로 바꿉니다. Pydantic이 응답 형식을 강제해,
들쭉날쭉한 LLM 응답을 안정적인 함수처럼 다룰 수 있게 해줍니다.

- 본인 영수증 사진이 있다면 아래에서 업로드하세요.
- 없다면 그대로 두면 저장소에 들어있는 샘플 영수증으로 진행됩니다.


In [ ]:
from pathlib import Path

USE_SAMPLE = True  # 직접 업로드하려면 False로 바꾸고 이 셀을 다시 실행하세요

if USE_SAMPLE:
    receipt_paths = [Path("receipts/sample_01.jpg")]
    print(f"샘플 영수증으로 진행합니다: {receipt_paths[0]}")
else:
    from google.colab import files

    uploaded = files.upload()
    receipt_paths = [Path(name) for name in uploaded]


In [ ]:
from budget_battle import client, vision

try:
    receipts = [vision.extract_receipt(path) for path in receipt_paths]
except (client.ApiCallFailed, RuntimeError) as error:
    show_api_hint(error)
    raise

for receipt in receipts:
    store = receipt.store or "매장 미상"
    print(f"{store} — {len(receipt.items)}개 항목, 합계 {receipt.total:,}원")


## 2막 — 정리한다: 가계부로 집계

추출한 항목들을 카테고리별로 모아 나만의 가계부 요약을 만듭니다. 이 구간은
API 호출 없이 순수 파이썬만으로 동작합니다.


In [ ]:
from budget_battle import ledger

book = ledger.build_ledger(receipts)
summary = ledger.summarize_for_agents(book)
print(summary)


## 3막 — 토론한다: 절약파 vs 플렉스파

같은 가계부를 두고 '자린고비 할머니'(절약파)와 'YOLO 인플루언서'(플렉스파)
두 AI가 토론합니다. 라운드 상한·호출 상한·합의 감지, 이 세 가지 종료 조건이
토론이 끝없이 이어지지 않게 막아줍니다.


In [ ]:
from budget_battle import client, debate, personas

try:
    turns = debate.run_debate(summary, personas.DEFAULT_PERSONAS)
except client.ApiCallFailed as error:
    show_api_hint(error)
    raise

for turn in turns:
    print(f"\n<{turn.speaker}>\n{turn.message}")


## 4막 — 판정한다: 재무 건강 점수

토론 전체를 읽은 판정관 AI가 재무 건강 점수와 오늘 당장 실행할 수 있는
처방을 내립니다.


In [ ]:
from budget_battle import client, judge

try:
    verdict = judge.judge_debate(summary, turns)
except client.ApiCallFailed as error:
    show_api_hint(error)
    raise

print(f"재무 건강 점수: {verdict.score}점")
print(verdict.diagnosis)
for number, prescription in enumerate(verdict.prescriptions, start=1):
    print(f"  {number}. {prescription}")


## 수고하셨습니다

여기까지 실행되면 오늘 실습을 끝까지 따라온 것입니다. 시간이 남으면 위
`USE_SAMPLE = True`를 `False`로 바꾸고 본인 영수증으로 다시 실행해
보세요.
